# 01 — Simulate + smoke test

Confirm the whole pipeline wires together on **mock** data (mock-local policy): simulate GReX +
trait with known ground truth, run the shared nested-CV harness for the **elastic-net** arm, and
(optionally, slower) the **Bayesian** arm. Reusable logic lives in `ptgs_bc`; this notebook only
imports and orchestrates.

## Setup

In [ ]:
import ptgs_bc as ptgs
from ptgs_bc import simulate_dataset, ElasticNetBuilder, BayesBuilder, nested_cv, run_benchmark, summary_table
print("ptgs_bc", ptgs.__version__)

## Simulate a dataset (known causal genes)

In [ ]:
ds, true_w = simulate_dataset(n_samples=400, n_genes=100, n_causal=15, family="gaussian", seed=0)
print(ds.n_samples, "samples ×", ds.n_genes, "genes | family:", ds.family)
ds.grex.iloc[:3, :5]

## Elastic-net baseline through nested CV

In [ ]:
res_en = nested_cv(ElasticNetBuilder(), ds, outer_k=5, seed=0)
print(res_en)

## Bayesian arm (small settings for a quick smoke run)

NUTS on ~100 genes is quick; at real gene counts raise warmup/samples or switch to SVI
(`BayesBuilder(inference="svi")`).

In [ ]:
bayes = BayesBuilder(prior="regularized_horseshoe", num_warmup=200, num_samples=200)
res_bayes = nested_cv(bayes, ds, outer_k=5, seed=0)
print(res_bayes)

## Head-to-head (same folds, same metric)

In [ ]:
res = run_benchmark([ElasticNetBuilder(), bayes], ds, outer_k=5, seed=0)
summary_table(res)